In [1]:
import os

print("DB host:", os.getenv("DATABASE_HOST"))
print("DB port:", os.getenv("DATABASE_PORT"))
print("DB name:", os.getenv("POSTGRES_DB"))
print("DB user:", os.getenv("POSTGRES_USER"))

DB host: postgres
DB port: 5432
DB name: de_workshop
DB user: de_user


In [2]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

db_user = os.environ["POSTGRES_USER"]
db_pass = os.environ["POSTGRES_PASSWORD"]   # not printed
db_name = os.environ["POSTGRES_DB"]
db_host = os.environ["DATABASE_HOST"]
db_port = os.environ["DATABASE_PORT"]

engine = create_engine(f"postgresql+psycopg2://{db_user}:{db_pass}@{db_host}:{db_port}/{db_name}")

with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database() AS db, current_user AS usr, now() AS ts;"))
    df = pd.DataFrame(result.fetchall(), columns=result.keys())

df

,db,usr,ts
0,de_workshop,de_user,2026-02-25 06:24:45.616610+00:00


In [3]:
import pandas as pd
import requests

api_url = "https://jsonplaceholder.typicode.com/posts"
response = requests.get(api_url, timeout=30)
response.raise_for_status()

df_posts = pd.DataFrame(response.json())
df_posts.head()

,userId,id,title,body
0,1,1,sunt aut facere repellat provident occaecati e...,quia et suscipit\nsuscipit recusandae consequu...
1,1,2,qui est esse,est rerum tempore vitae\nsequi sint nihil repr...
2,1,3,ea molestias quasi exercitationem repellat qui...,et iusto sed quo iure\nvoluptatem occaecati om...
3,1,4,eum et est occaecati,ullam et saepe reiciendis voluptatem adipisci\...
4,1,5,nesciunt quas odio,repudiandae veniam quaerat sunt sed\nalias aut...


In [4]:
df_clean = df_posts.copy()

# Standardize column names (enterprise habit)
df_clean.columns = [c.strip().lower() for c in df_clean.columns]

# Enforce types (helps consistency in downstream systems)
df_clean["userid"] = df_clean["userid"].astype("int64")
df_clean["id"] = df_clean["id"].astype("int64")
df_clean["title"] = df_clean["title"].astype("string")
df_clean["body"] = df_clean["body"].astype("string")

# Basic data quality checks (simple but valuable)
null_counts = df_clean.isna().sum()
row_count = len(df_clean)

row_count, null_counts.to_dict()


(100, {'userid': 0, 'id': 0, 'title': 0, 'body': 0})

In [5]:
from sqlalchemy import text

table_name = "raw_posts"

# Load (lab-safe default: replace the table each run)
# In production you'd usually do staging + swap, but replace is fine for learning.
df_clean.to_sql(
    name=table_name,
    con=engine,
    if_exists="replace",
    index=False,
    method="multi",
    chunksize=1000,
)

# Verify
with engine.connect() as conn:
    count = conn.execute(text(f"SELECT COUNT(*) FROM {table_name};")).scalar()

count

100